In [3]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix

In [5]:
weights_dir = "../models/roberta_baseline"

# Load trained model
tokenizer = AutoTokenizer.from_pretrained(weights_dir)
model = AutoModelForSequenceClassification.from_pretrained(weights_dir)

raw_dataset = load_dataset("imdb")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1196.95it/s]


In [6]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding='max_length', truncation=True, max_lengh=128)

tokenized_datasets = raw_dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(2000))

batch_size = 64
eval_dataloader = DataLoader(eval_dataset, batch_size=batch_size)

Map: 100%|██████████| 50000/50000 [00:11<00:00, 4300.65 examples/s]


In [7]:
device = torch.device("cuda")
model.to(device)

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
	for batch in eval_dataloader:
		batch = {k: v.to(device) for k, v in batch.items()}
		outputs = model(**batch)
		
		# Logits are the raw scores
		logits = outputs.logits
		
		# Taking the index of the highest score (0 or 1)
		predictions = torch.argmax(logits, dim=-1)
		
		# Moving back to CPU and converting to numpy arrays
		all_preds.extend(predictions.cpu().numpy())
		all_labels.extend(batch["labels"].cpu().numpy())

In [8]:
macro_f1 = f1_score(all_labels, all_preds, average="macro")
print(f"Macro-F1: {macro_f1:.4f}")

cm = confusion_matrix(all_labels, all_preds)

# Calculate TPR for class 0
# cm[0, 0] = Actually 0, Predicted 0 (Correct)
# cm[0, 1] = Actually 0, Predicted 1 (Incorrect)
tpr_class_0 = cm[0, 0] / max(cm[0, 0] + cm[0, 1], 0)

# Calculate TPR for class 1
# cm[1, 0] = Actually 1, Predicted 0 (Incorrect)
# cm[1, 1] = Actually 1, Predicted 1 (Correct)
tpr_class_1 = cm[1, 1] / max(cm[1, 1] + cm[1, 0], 0)

tpr_gap = abs(tpr_class_0 - tpr_class_1)

print(f"TPR Class 0: {tpr_class_0:.4f}")
print(f"TPR Class 1: {tpr_class_1:.4f}")
print(f"TPRGap: {tpr_gap:.4f}")

Macro-F1: 0.8136
TPR Class 0: 0.9890
TPR Class 1: 0.6490
TPRGap: 0.3400
